# 5.1 瑞利近似下的反射率

**学习目标**
- 理解瑞利散射近似下反射率因子的计算
- 掌握Z和ZDR与滴谱分布的关系
- 观察不同波段对Z和ZDR计算的影响

**参考文献**：Ryzhkov & Zrnic, Chapter 5, pp. 461-500

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown
import ipywidgets as widgets
plt.rcParams['font.family'] = ['DejaVu Sans', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
def marshall_palmer_dsd(D_mm, R):
    """Marshall-Palmer 滴谱分布"""
    N0 = 8e3  # mm^-1 m^-3
    Lambda = 4.1 * R**(-0.21)  # mm^-1
    return N0 * np.exp(-Lambda * D_mm)

def rayleigh_reflectivity(D_mm, wavelength_mm, K_squared=0.93):
    """瑞利散射截面"""
    sigma = (np.pi**5 / wavelength_mm**4) * K_squared * D_mm**6
    return sigma

def calculate_Z_ZDR(R, wavelength_mm=10.7, K_squared=0.93):
    """
    计算反射率和差分反射率
    简化模型：假设雨滴为椭球
    """
    D_range = np.linspace(0.1, 8, 200)
    dD = D_range[1] - D_range[0]
    
    # 滴谱
    N_D = marshall_palmer_dsd(D_range, R)
    
    # 散射截面
    sigma = rayleigh_reflectivity(D_range, wavelength_mm, K_squared)
    
    # 雨滴轴比
    axis_ratio = np.maximum(1 - 0.05*D_range, 0.6)
    
    # 水平和垂直通道的散射（简化）
    # 假设粒子主轴垂直取向
    f_h = axis_ratio**2  # 水平偏振权重
    f_v = np.ones_like(axis_ratio)  # 垂直偏振权重
    
    # 积分计算Z
    Z_h = np.sum(N_D * sigma * f_h * D_range**6) * dD * 1e-3
    Z_v = np.sum(N_D * sigma * f_v * D_range**6) * dD * 1e-3
    
    # 转换为dBZ
    Z_h_dbz = 10 * np.log10(Z_h + 1e-10)
    Z_v_dbz = 10 * np.log10(Z_v + 1e-10)
    ZDR = Z_h_dbz - Z_v_dbz
    
    return Z_h_dbz, Z_v_dbz, ZDR

def compare_bands():
    """对比不同波段的Z和ZDR"""
    wavelengths = {'X-band (3.2cm)': 3.2, 'C-band (5.4cm)': 5.4, 'S-band (10.7cm)': 10.7}
    R_range = np.linspace(0.5, 100, 100)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for band, wl_mm in wavelengths.items():
        Z_range = []
        ZDR_range = []
        for R in R_range:
            Z_h, Z_v, ZDR = calculate_Z_ZDR(R, wl_mm)
            Z_range.append(Z_h)
            ZDR_range.append(ZDR)
        
        axes[0].semilogy(R_range, [10**(z/10) for z in Z_range], linewidth=2, label=band)
        axes[1].plot(R_range, ZDR_range, linewidth=2, label=band)
    
    axes[0].set_xlabel('降雨率 R (mm/h)', fontsize=11)
    axes[0].set_ylabel('Z (mm⁶/m³)', fontsize=11)
    axes[0].set_title('R-Z 关系 (对数坐标)', fontsize=12)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].set_xlabel('降雨率 R (mm/h)', fontsize=11)
    axes[1].set_ylabel('ZDR (dB)', fontsize=11)
    axes[1].set_title('R-ZDR 关系', fontsize=12)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== 各波段典型值对比 ===")
    print(f"{'R (mm/h)':<10} {'X-band Z (dBZ)':<18} {'C-band Z (dBZ)':<18} {'S-band Z (dBZ)':<18} {'ZDR (dB)':<12}")
    print("-" * 76)
    for R in [1, 5, 10, 20, 50, 100]:
        Zx, _, ZDRx = calculate_Z_ZDR(R, 3.2)
        Zc, _, ZDRc = calculate_Z_ZDR(R, 5.4)
        Zs, _, ZDRs = calculate_Z_ZDR(R, 10.7)
        print(f"{R:<10} {Zx:<18.1f} {Zc:<18.1f} {Zs:<18.1f} {ZDRs:<12.2f}")

In [ ]:
compare_bands()

## 练习题

1. **R-Z关系**：降雨率从1增加到50mm/h时，反射率Z增加多少dB？

2. **波段差异**：为什么S波段的反射率值比X波段大？

3. **ZDR随R增加**：ZDR随降雨率增加而增大，解释其物理原因。

## 参考文献

- Ryzhkov, A.V., and D.S. Zrnic, 2019: *Radar Polarimetry for Weather Observations*, Chapter 5, Springer.